## Setup
Imports and device selection (GPU if available, otherwise CPU).

In [ ]:
import torch                          # core PyTorch library
import torch.nn as nn                 # neural network building blocks (layers, losses)
import torch.optim as optim           # optimizers (we'll use Adam)
import numpy as np                    # numerical arrays, used for the board representation
import random                         # for epsilon-greedy exploration and replay sampling
from collections import deque         # fixed-size ring buffer, used for the replay memory

if torch.cuda.is_available():
    device = torch.device("cuda")     # use NVIDIA GPU if one is present
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")      # otherwise fall back to Intel XPU if available
else:
    device = torch.device("cpu")      # otherwise just run on CPU

print("Using device:", device)        # confirm which device training will run on


## Environment
A minimal tic-tac-toe game engine: tracks the board, whose turn it is, and reports rewards/termination back to the agent.

In [ ]:
class TicTacToe:
    # n=3 for normal tic tac toe, could probably generalize this more but whatever
    def __init__(self, n=3):
        self.n = n                              # board dimension (3 -> 3x3 board)
        self.board = np.zeros(n*n)              # flat array of 9 cells, 0 = empty
        self.current_player = 1                 # player 1 (X) always starts

    def reset(self):
        self.board = np.zeros(self.n*self.n)    # clear the board back to all zeros
        self.current_player = 1                 # reset turn order to player 1 (X)
        return self.board.copy()                # return a fresh copy of the starting state

    def available_actions(self):
        # indices where board is still 0
        avail = []                              # will collect indices of empty cells
        for i in range(len(self.board)):
            if self.board[i] == 0:
                avail.append(i)                 # cell i is empty, so it's a legal move
        return avail                            # list of legal move indices

    def step(self, action):
        if self.board[action] != 0:
            print("tried an invalid move??", action)   # sanity check, shouldn't happen if masking works
            return self.board.copy(), -10, True         # penalize and end episode on illegal move

        self.board[action] = self.current_player        # place the current player's mark

        if self.check_win(self.current_player):
            return self.board.copy(), 10, True           # current player just won: +10 reward, episode ends

        if len(self.available_actions()) == 0:
            # board full, no winner = draw
            return self.board.copy(), 0, True            # no cells left and no winner: draw, 0 reward, episode ends

        self.current_player *= -1                        # flip turn: 1 -> -1 or -1 -> 1
        return self.board.copy(), 0, False               # game continues, no reward yet

    def check_win(self, p):
        b = self.board.reshape(self.n, self.n)   # reshape flat 9-array into a 3x3 grid for easy row/col/diag checks

        for i in range(self.n):
            if np.all(b[i, :] == p):
                return True                      # row i is all player p's marks
            if np.all(b[:, i] == p):
                return True                      # column i is all player p's marks

        # diagonals
        if np.all(np.diag(b) == p):
            return True                          # main diagonal (top-left to bottom-right) is all p
        if np.all(np.diag(np.fliplr(b)) == p):
            return True                          # anti-diagonal (top-right to bottom-left) is all p

        return False                             # no winning line found for player p

    def render(self):
        # quick debug helper, not really used in training
        symbols = {1: 'X', -1: 'O', 0: '.'}      # map numeric board values to printable symbols
        b = self.board.reshape(self.n, self.n)   # reshape to 3x3 for row-by-row printing
        for row in b:
            print(' '.join(symbols[int(x)] for x in row))   # print each row as space-separated symbols


## Replay Memory
Stores past transitions `(state, action, next_state, reward, done)` so the network can train on randomly sampled batches instead of only the most recent move (this breaks correlation between consecutive updates).

In [ ]:
from collections import namedtuple

Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward', 'done'))   # one experience tuple

class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)   # ring buffer: oldest transitions drop off once full

    def push(self, *args):
        self.memory.append(Transition(*args))      # store a new transition

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)   # randomly sample a batch (decorrelates training data)

    def __len__(self):
        return len(self.memory)                    # current number of stored transitions

memory = ReplayMemory(5000)                         # replay buffer holding up to 5,000 transitions


## The Network (DQN)
A simple 3-layer MLP: takes the 9-cell board as input and outputs one Q-value per cell (the estimated value of playing there).

In [ ]:
class DQN(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.layer1 = nn.Linear(size, 256)   # input layer: board size (9) -> 256 hidden units
        self.layer2 = nn.Linear(256, 256)    # hidden layer: 256 -> 256
        self.layer3 = nn.Linear(256, size)   # output layer: 256 -> 9 Q-values, one per cell
        self.relu = nn.ReLU()                # shared ReLU activation used after layer1 and layer2


    def forward(self, x):
        x = self.relu(self.layer1(x))        # apply first linear layer, then ReLU
        x = self.relu(self.layer2(x))        # apply second linear layer, then ReLU
        return self.layer3(x)                # final linear layer, no activation -> raw Q-values


## Agent Setup
Instantiate the online network and a target network, set up the optimizer and DQN hyperparameters, and define the helper functions used during training: canonical state conversion, action selection, and the training step itself.

In [ ]:
board_size = 9
model = DQN(board_size).to(device)                          # the network actually being trained
target_model = DQN(board_size).to(device)                   # a slowly-updated copy used to compute stable targets
target_model.load_state_dict(model.state_dict())            # start target network with the same weights as model
optimizer = optim.Adam(model.parameters(), lr=1e-3)          # Adam optimizer, learning rate 0.001

gamma = 0.99            # discount factor: how much future reward matters vs immediate reward
eps = 1.0                # exploration rate: starts fully random...
eps_min = 0.05            # ...and never decays below this floor
eps_decay = 0.9995        # eps is multiplied by this after every training step
tau = 0.01  # soft update rate for target network, applied periodically during training


def get_canonical_state(board, player):
    # network always sees the board as "my pieces = 1, opponent = -1", regardless
    # of whether we're actually player 1 or player -1. Without this, the same
    # network has to separately learn how to play as X and as O.
    return board * player            # flips the sign of every cell if we're player -1


def select_action(canonical_state, available_actions, current_eps):
    if np.random.rand() <= current_eps:
        return random.choice(available_actions)          # explore: pick a random legal move

    state_t = torch.FloatTensor(canonical_state).unsqueeze(0).to(device)   # convert state to a batch of size 1
    with torch.no_grad():
        q_values = model(state_t)                         # forward pass: get Q-value for every cell

    # mask out illegal moves so argmax cant pick them
    mask = torch.full((board_size,), -float('inf'), device=device)   # start with all moves "impossible"
    mask[available_actions] = 0                            # un-mask only the legal moves (add 0, i.e. no penalty)
    return torch.argmax(q_values + mask).item()             # exploit: pick the legal move with highest Q-value


def train_step(batch_size):
    global eps

    if len(memory) < batch_size:
        return                                              # not enough data yet, skip this training step

    batch = memory.sample(batch_size)                        # randomly sample a batch of past transitions

    states = torch.FloatTensor(np.array([t.state for t in batch])).to(device)        # batch of starting states
    actions = torch.LongTensor([t.action for t in batch]).to(device)                 # batch of actions taken
    rewards = torch.FloatTensor([t.reward for t in batch]).to(device)                # batch of rewards received
    next_states = torch.FloatTensor(np.array([t.next_state for t in batch])).to(device)  # batch of resulting states
    dones = torch.FloatTensor([float(t.done) for t in batch]).to(device)             # batch of episode-end flags

    # Q(s, a) for the actions actually taken -- one batched forward pass
    q_values = model(states).gather(1, actions.unsqueeze(1)).squeeze(1)   # pick out the Q-value for the action taken

    # next_state is stored canonically from the OPPONENT's perspective (it's their
    # turn to move next). Tic-tac-toe is zero-sum, so their best move is bad for
    # us -> subtract their max Q instead of adding it. Use target_model here for
    # stable bootstrapping targets.
    with torch.no_grad():
        next_q = target_model(next_states).max(1)[0]                     # opponent's best available Q-value, next turn
        targets = rewards - gamma * next_q * (1.0 - dones)               # Bellman target, subtracting opponent's value

    loss = nn.SmoothL1Loss()(q_values, targets)     # Huber loss between predicted and target Q-values
    optimizer.zero_grad()                            # clear gradients from the previous step
    loss.backward()                                  # backpropagate the loss
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # clip gradients to avoid unstable updates
    optimizer.step()                                  # apply the optimizer update to the network weights

    if eps > eps_min:
        eps *= eps_decay                              # decay exploration rate a little after each training step


## Training Loop
Play the agent against itself for `episodes` games, storing every transition and training on a batch after every single move. The target network is soft-updated periodically for stability.

In [ ]:
N = 3
env = TicTacToe(n=N)                    # create the game environment (3x3 board)
episodes = 5000  # tuned to finish in ~1-2 min; ~2000 episodes already hits ~97% winrate vs random
batch_size = 64                         # number of transitions sampled per training step
target_update_every = 20  # episodes, soft-updates target_model throughout training

for e in range(episodes):
    state = env.reset()                 # start a fresh game, get the empty board

    for t in range(N*N):                # at most 9 moves possible in a game
        player = env.current_player                        # whose turn is it right now
        canon_state = get_canonical_state(state, player)    # view the board from this player's perspective
        actions = env.available_actions()                   # legal moves in the current state
        action = select_action(canon_state, actions, eps)    # choose a move (explore or exploit)

        next_state, reward, done = env.step(action)          # apply the move, get reward and done flag
        # next mover is env.current_player after step (unchanged if the game ended)
        canon_next_state = get_canonical_state(next_state, env.current_player)   # next state, from the NEXT player's perspective
        memory.push(canon_state, action, canon_next_state, reward, done)         # store this transition in replay memory

        train_step(batch_size)  # train on every move, not just once per episode

        state = next_state                # advance to the new board state
        if done:
            break                          # game over, stop the inner move loop

    if (e + 1) % target_update_every == 0:
        for tp, p in zip(target_model.parameters(), model.parameters()):
            tp.data.copy_(tau * p.data + (1 - tau) * tp.data)   # soft update: nudge target weights toward model weights

    if (e+1) % 500 == 0:
        print(f"Episode: {e+1}/{episodes}, Epsilon: {eps:.2f}")   # progress update every 500 episodes

print(f"done training, N={N}")           # training finished


## Evaluation
Play the trained agent (no exploration, `eps=0`) against a purely random opponent over many games and report the win/draw/loss rate.

In [ ]:
def evaluate(env, num_games=100):
    w = 0                                # win counter
    d = 0                                # draw counter
    l = 0                                # loss counter

    for g in range(num_games):
        state = env.reset()              # start a fresh game
        done = False

        while not done:
            player = env.current_player                       # our trained agent always moves first here
            canon_state = get_canonical_state(state, player)   # canonical view of the board for the agent
            actions = env.available_actions()                  # legal moves available
            action = select_action(canon_state, actions, current_eps=0)   # eps=0: always exploit, no randomness
            state, reward, done = env.step(action)              # agent makes its move

            if done:
                if reward == 10:
                    w += 1                # agent won
                elif reward == 0:
                    d += 1                # game ended in a draw
                break

            # opponent just plays randomly for now
            actions = env.available_actions()                   # recompute legal moves for the opponent
            action = random.choice(actions)                     # opponent picks a uniformly random legal move
            state, reward, done = env.step(action)               # opponent makes its move

            if done:
                if reward == 10:
                    l += 1                # opponent won -> a loss for our agent
                elif reward == 0:
                    d += 1                # draw
                break

    print(f'over {num_games} games vs random:')
    print(f'wins: {w} ({w/num_games*100:.1f}%)')      # agent's win rate
    print(f'draws: {d} ({d/num_games*100:.1f}%)')     # draw rate
    print(f'losses: {l} ({l/num_games*100:.1f}%)')    # loss rate

evaluate(env)                                          # run the evaluation with default 100 games


## Save the Trained Model

In [ ]:
torch.save(model.state_dict(), "best_model.pth")   # save only the learned weights (state_dict), not the full model object


In [ ]:
import os
os.listdir()   # confirm best_model.pth was written to the current directory


## Download (Colab only)

In [ ]:
from google.colab import files   # Colab-specific helper for triggering a browser download
files.download("best_model.pth")   # download the trained weights to your local machine
